# Step 3 — Query→Product Text Relevance (TF-IDF)

**Purpose.** Close the main gap found in Step 2: the query features contributed nothing to the ranker (±0.000 NDCG@10 in the ablation), even though 62% of test queries contain product-attribute terms (interim analysis) and **80.5% of candidates are cold** — no user history, so text relevance is the main signal available for them.

This notebook adds four pair-level text features, evaluates them alone and inside the LightGBM benchmark, and checks specifically whether they help on cold candidates.

| Feature | Definition |
|---|---|
| `q_name_tfidf_sim` | TF-IDF cosine between the query terms and the product **name** terms |
| `q_name_match_frac` | share of query terms that appear in the product name |
| `q_name_match_idf` | IDF-weighted version of the above (rare matched terms count more) |
| `q_meta_tfidf_sim` | TF-IDF cosine between the query and the product's **brand + category names** |

**Inputs:** `candidates/users/queries/features.parquet` from Steps 1–2 + `product_meta_data.txt` (text columns).
**Output:** `features.parquet` updated in place with the four new columns (idempotent — re-running just recomputes them).
**Runtime:** ~15 min; peak ~8 GB RAM (streamed parsing + chunked sparse products keep it bounded).

In [1]:
import os, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pyarrow as pa, pyarrow.compute as pc, pyarrow.csv as pacsv

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'
SEED = 42

cands = pd.read_parquet(OUT_DIR / 'candidates.parquet', columns=['user', 'wid', 'label'])
users = pd.read_parquet(OUT_DIR / 'users.parquet')
queries = pd.read_parquet(OUT_DIR / 'queries.parquet')
nU = len(users)
cand_wids = np.unique(cands['wid'].to_numpy())
print(f'candidate rows {len(cands):,} | users {nU:,} | unique candidate products {len(cand_wids):,}')

Mounted at /content/drive
candidate rows 15,510,012 | users 173,831 | unique candidate products 8,362,535


## 1. Parse the product text for candidate products only

All text in JDSearch is anonymized **term IDs** (integers) separated by `\x18`, shared between queries and product text — which is exactly what makes lexical matching possible without knowing the words. Only the 8.36M products that ever appear in a candidate slate are needed, so the 4.8 GB metadata file is streamed in batches and filtered on the fly; term lists are kept as flat int arrays (never Python lists per row).

The **name** forms one document space; **brand_name + cate_name_2/3/4** form a second, coarser one. `-1` placeholder tokens (empty fields) are dropped.

In [2]:
NAME_COLS, META_COLS = ['name'], ['brand_name', 'cate_name_2', 'cate_name_3', 'cate_name_4']
cols = ['wid'] + NAME_COLS + META_COLS

def tokens_of(col):
    """Arrow string column -> (tokens per row, flat int64 tokens); empty fields become -1."""
    parts = pc.split_pattern(pc.fill_null(col, '-1'), '\x18')
    flat = pc.list_flatten(parts)
    flat = pc.if_else(pc.equal(flat, ''), '-1', flat)
    return pc.list_value_length(parts).to_numpy().astype(np.int64), pc.cast(flat, pa.int64()).to_numpy()

t0 = time.time()
wid_parts, name_parts, meta_parts = [], [], []
reader = pacsv.open_csv(DATA_DIR / 'product_meta_data.txt',
    parse_options=pacsv.ParseOptions(delimiter='\t', quote_char=False),
    convert_options=pacsv.ConvertOptions(include_columns=cols,
        column_types={'wid': pa.int64(), **{c: pa.string() for c in NAME_COLS + META_COLS}}))
off = 0
for batch in reader:
    tb = pa.Table.from_batches([batch])
    wid = tb['wid'].to_numpy()
    j = np.searchsorted(cand_wids, wid)
    keep = (j < len(cand_wids)) & (cand_wids[np.minimum(j, len(cand_wids) - 1)] == wid)
    if not keep.any():
        continue
    tb = tb.filter(pa.array(keep))
    ids = np.arange(off, off + len(tb), dtype=np.int64); off += len(tb)
    wid_parts.append(tb['wid'].to_numpy())
    ln, tk = tokens_of(tb.column('name'))
    name_parts.append((np.repeat(ids, ln), tk))
    for c in META_COLS:
        ln, tk = tokens_of(tb.column(c))
        meta_parts.append((np.repeat(ids, ln), tk))

doc_wids = np.concatenate(wid_parts); del wid_parts
def flat(parts):
    d = np.concatenate([p[0] for p in parts]); t = np.concatenate([p[1] for p in parts])
    m = t != -1
    return d[m], t[m]
name_doc, name_tok = flat(name_parts); del name_parts
meta_doc, meta_tok = flat(meta_parts); del meta_parts
gc.collect()
nD = len(doc_wids)
print(f'parsed in {time.time() - t0:.0f}s: {nD:,} candidate products with metadata '
      f'({nD / len(cand_wids):.1%}) | name tokens {len(name_tok):,} | meta tokens {len(meta_tok):,}')

parsed in 66s: 8,305,622 candidate products with metadata (99.3%) | name tokens 234,857,084 | meta tokens 67,625,508


## 2. TF-IDF document and query matrices

Binary TF (a term either occurs in a short text or not) with smoothed IDF, L2-normalised rows — so each pair score below is a plain sparse dot product. Query terms that never occur in any product text still count toward the query norm (they *should* lower the similarity: the product cannot match them), and the query matrices for the match features are built in the same space:

- `Q_frac` rows hold `1 / n_query_terms` → dot with a binary document gives the **share of query terms matched**.
- `Q_idf` rows hold `idf / Σ idf` → dot gives the **IDF-weighted share**.

In [3]:
# query tokens (test query of each user)
qtext = users['qry'].map(queries.set_index('qry')['text'])
qsplit = qtext.str.split('\x18')
q_doc = np.repeat(np.arange(nU, dtype=np.int64), qsplit.str.len().to_numpy())
q_tok = pd.to_numeric(qsplit.explode(), errors='coerce').fillna(-1).to_numpy(np.int64)
q_doc, q_tok = q_doc[q_tok != -1], q_tok[q_tok != -1]

def dedupe(doc, code, V):
    key = np.unique(doc * V + code)
    return key // V, (key % V).astype(np.int64)

def build_space(doc, tok, n_docs):
    """-> (normalised doc CSR, vocab, idf). Binary tf, idf = log((N+1)/(df+1)) + 1."""
    vocab = np.unique(tok)
    d, c = dedupe(doc, np.searchsorted(vocab, tok), len(vocab))
    df = np.bincount(c, minlength=len(vocab))
    idf = (np.log((n_docs + 1) / (df + 1)) + 1).astype(np.float32)
    norm = np.sqrt(np.bincount(d, weights=idf[c] ** 2, minlength=n_docs)); norm[norm == 0] = 1
    D = sp.csr_matrix((idf[c] / norm[d], (d, c)), shape=(n_docs, len(vocab)), dtype=np.float32)
    return D, vocab, idf

def query_matrices(vocab, idf, with_match=False):
    """Query rows in the doc space; out-of-vocab terms add to the norm but can never match."""
    d, c_raw = dedupe(q_doc, q_tok, q_tok.max() + 1)          # dedupe terms within each query
    pos = np.searchsorted(vocab, c_raw)
    in_v = (pos < len(vocab)) & (vocab[np.minimum(pos, len(vocab) - 1)] == c_raw)
    idf_oov = np.float32(np.log(nD + 1) + 1)                  # df = 0
    w = np.where(in_v, idf[np.minimum(pos, len(vocab) - 1)], idf_oov)
    norm = np.sqrt(np.bincount(d, weights=w ** 2, minlength=nU)); norm[norm == 0] = 1
    Q = sp.csr_matrix((w[in_v] / norm[d[in_v]], (d[in_v], pos[in_v])), shape=(nU, len(vocab)), dtype=np.float32)
    if not with_match:
        return Q
    n_terms = np.bincount(d, minlength=nU).astype(np.float32); n_terms[n_terms == 0] = 1
    idf_sum = np.bincount(d, weights=w, minlength=nU).astype(np.float32); idf_sum[idf_sum == 0] = 1
    Qf = sp.csr_matrix((1 / n_terms[d[in_v]], (d[in_v], pos[in_v])), shape=Q.shape, dtype=np.float32)
    Qi = sp.csr_matrix((w[in_v] / idf_sum[d[in_v]], (d[in_v], pos[in_v])), shape=Q.shape, dtype=np.float32)
    return Q, Qf, Qi

D_name, vocab_n, idf_n = build_space(name_doc, name_tok, nD); del name_doc, name_tok
D_meta, vocab_m, idf_m = build_space(meta_doc, meta_tok, nD); del meta_doc, meta_tok
Q_name, Q_frac, Q_idf = query_matrices(vocab_n, idf_n, with_match=True)
Q_meta = query_matrices(vocab_m, idf_m)
D_bin = sp.csr_matrix((np.ones_like(D_name.data), D_name.indices, D_name.indptr), shape=D_name.shape)
gc.collect()
print(f'name space: {D_name.shape[0]:,} docs x {D_name.shape[1]:,} terms ({D_name.nnz:,} nnz) | '
      f'meta space: {D_meta.shape[1]:,} terms ({D_meta.nnz:,} nnz)')

name space: 8,305,622 docs x 802,096 terms (195,900,016 nnz) | meta space: 183,214 terms (50,518,396 nnz)


## 3. Score all 15.5M (query, candidate) pairs

Chunked row-pair dot products: gather the matching query rows and document rows, multiply elementwise, sum. Candidates without metadata (0.45%) get `NaN`, which LightGBM handles natively.

In [4]:
doc_of = pd.Index(doc_wids).get_indexer(cands['wid'])          # -1 = no metadata
qi_all, ok = cands['user'].to_numpy(), doc_of >= 0
qi, di = qi_all[ok], doc_of[ok]

def rowpair_dot(Q, D, chunk=2_000_000):
    out = np.empty(len(qi), np.float32)
    for s in range(0, len(qi), chunk):
        e = min(s + chunk, len(qi))
        out[s:e] = np.asarray(Q[qi[s:e]].multiply(D[di[s:e]]).sum(axis=1)).ravel()
    return out

t0 = time.time()
TEXT_FEATURES = ['q_name_tfidf_sim', 'q_name_match_frac', 'q_name_match_idf', 'q_meta_tfidf_sim']
text = {f: np.full(len(cands), np.nan, np.float32) for f in TEXT_FEATURES}
text['q_name_tfidf_sim'][ok] = rowpair_dot(Q_name, D_name)
text['q_name_match_frac'][ok] = rowpair_dot(Q_frac, D_bin)
text['q_name_match_idf'][ok] = rowpair_dot(Q_idf, D_bin)
text['q_meta_tfidf_sim'][ok] = rowpair_dot(Q_meta, D_meta)
del D_name, D_meta, D_bin, Q_name, Q_frac, Q_idf, Q_meta; gc.collect()
print(f'scored {len(qi):,} pairs in {time.time() - t0:.0f}s')
pd.DataFrame(text).describe().T.round(4)

scored 15,439,674 pairs in 25s


,count,mean,std,min,25%,50%,75%,max
q_name_tfidf_sim,15439674.0,0.2376,0.1345,0.0,0.1501,0.2459,0.3243,1.0
q_name_match_frac,15439674.0,0.7209,0.3438,0.0,0.5000,1.0000,1.0000,1.0
q_name_match_idf,15439674.0,0.7109,0.3566,0.0,0.4472,1.0000,1.0000,1.0
q_meta_tfidf_sim,15439674.0,0.1579,0.2019,0.0,0.0000,0.0000,0.2656,1.0


## 4. Do the features carry signal on their own?

Same per-slate evaluation as Step 2 (random tie-breaking, gain 2^label − 1), each text feature used alone as a ranker on the validation slates.

In [5]:
def evaluate(df, score, k=10, seed=0):
    """Vectorised per-slate ranking metrics with random tie-breaking (identical to Step 2)."""
    u = df['user'].to_numpy(); lab = df['label'].to_numpy().astype(np.float64)
    r = np.random.default_rng(seed).random(len(df))
    order = np.lexsort((r, -np.asarray(score, dtype=np.float64), u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    rank = np.arange(len(u_s)) - np.repeat(start, np.diff(np.r_[start, len(u_s)]))
    top = rank < k
    disc = np.where(top, 1.0 / np.log2(rank + 2), 0.0)
    gid = np.repeat(np.arange(len(start)), np.diff(np.r_[start, len(u_s)]))
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    ideal_order = np.lexsort((-lab, u)); lab_i = lab[ideal_order]
    idcg = np.bincount(gid, weights=(2 ** lab_i - 1) * disc)
    has_pos = np.bincount(gid, weights=(lab_s > 0)) > 0
    has_buy = np.bincount(gid, weights=(lab_s == 3)) > 0
    hit = np.bincount(gid, weights=((lab_s > 0) & top)) > 0
    hit_buy = np.bincount(gid, weights=((lab_s == 3) & top)) > 0
    return {'NDCG@10': (dcg[has_pos] / idcg[has_pos]).mean(), 'HR@10': hit[has_pos].mean(),
            'Purchase-HR@10': hit_buy[has_buy].mean(), 'slates': int(has_pos.sum())}

split = pd.read_parquet(OUT_DIR / 'features.parquet', columns=['split'])['split'].to_numpy()
assert len(split) == len(cands)
ev_df = pd.DataFrame({'user': cands['user'], 'label': cands['label']})
val_m = split == 'val'
single = pd.DataFrame({f: evaluate(ev_df[val_m], pd.Series(text[f][val_m]).fillna(0)) for f in TEXT_FEATURES}).T
single.loc['random'] = evaluate(ev_df[val_m], np.zeros(int(val_m.sum())))
single.sort_values('NDCG@10', ascending=False).round(4)

,NDCG@10,HR@10,Purchase-HR@10,slates
q_name_tfidf_sim,0.0960,0.2444,0.1563,25825.0
q_name_match_idf,0.0942,0.2430,0.1626,25825.0
q_name_match_frac,0.0939,0.2424,0.1629,25825.0
q_meta_tfidf_sim,0.0852,0.2272,0.1399,25825.0
random,0.0846,0.2326,0.1563,25825.0


## 5. Attach to the feature table and retrain the benchmark

Identical training setup to Step 2 (same seed, same user-level split, slates shuffled, LambdaRank, early stopping on validation NDCG@10) — one model on the 42 Step-2 features, one with the 4 text features added.

In [6]:
feat = pd.read_parquet(OUT_DIR / 'features.parquet')
for f in TEXT_FEATURES:
    feat[f] = text[f]
del text; gc.collect()
feat.to_parquet(OUT_DIR / 'features.parquet', index=False)
FEATURES_ALL = [c for c in feat.columns if c not in ('user', 'wid', 'label', 'split')]
FEATURES_BASE = [c for c in FEATURES_ALL if c not in TEXT_FEATURES]
print(f'saved features.parquet with {len(FEATURES_ALL)} features '
      f'({len(FEATURES_BASE)} base + {len(TEXT_FEATURES)} text)')

saved features.parquet with 46 features (42 base + 4 text)


In [7]:
import lightgbm as lgb, warnings
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated')

def prepared(split_name):
    d = feat[feat['split'] == split_name]
    d = d.iloc[np.lexsort((np.random.default_rng(SEED).random(len(d)), d['user'].to_numpy()))]
    d = d[d.groupby('user')['label'].transform('max') > 0]
    return d, d.groupby('user', sort=False).size().to_numpy()

tr, g_tr = prepared('train'); va, g_va = prepared('val'); te, _ = prepared('test')
del feat; gc.collect()

def fit(cols, n_estimators=600):
    m = lgb.LGBMRanker(objective='lambdarank', n_estimators=n_estimators, learning_rate=0.05, num_leaves=63,
                       min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                       random_state=SEED, n_jobs=-1, verbose=-1)
    m.fit(tr[cols], tr['label'], group=g_tr, eval_set=[(va[cols], va['label'])], eval_group=[g_va],
          eval_at=[10], callbacks=[lgb.early_stopping(50, verbose=False)])
    return m

t0 = time.time(); base = fit(FEATURES_BASE); print(f'base model: {time.time() - t0:.0f}s, iter {base.best_iteration_}')
t0 = time.time(); full = fit(FEATURES_ALL); print(f'+text model: {time.time() - t0:.0f}s, iter {full.best_iteration_}')

results = pd.DataFrame({
    'random': evaluate(te, np.zeros(len(te))),
    'best single text feature (q_name_tfidf_sim)': evaluate(te, te['q_name_tfidf_sim'].fillna(0)),
    'LightGBM base (42 window-aware features)': evaluate(te, base.predict(te[FEATURES_BASE])),
    'LightGBM + text features': evaluate(te, full.predict(te[FEATURES_ALL])),
}).T
results.round(4)

base model: 196s, iter 320
+text model: 240s, iter 386


,NDCG@10,HR@10,Purchase-HR@10,slates
random,0.0839,0.2268,0.1371,25822.0
best single text feature (q_name_tfidf_sim),0.0955,0.2440,0.1468,25822.0
LightGBM base (42 window-aware features),0.4075,0.7046,0.6633,25822.0
LightGBM + text features,0.4144,0.7139,0.6695,25822.0


In [8]:
imp = pd.Series(full.booster_.feature_importance('gain'), index=FEATURES_ALL)
print('share of total split gain taken by the text features:')
(imp / imp.sum()).loc[TEXT_FEATURES].sort_values(ascending=False).round(4)

share of total split gain taken by the text features:


,0
q_name_tfidf_sim,0.0382
q_name_match_frac,0.0078
q_meta_tfidf_sim,0.0043
q_name_match_idf,0.0037


## 6. Where does the text help? Cold-candidate segment

A candidate is **cold** when its product never appears in any user's history — no `uc_*` or `i_users_*` feature can see it. Slates are grouped by whether their positives are all-cold, mixed, or all-warm, and both models are compared on each segment.

In [9]:
hist_wids = np.unique(pd.read_parquet(OUT_DIR / 'events.parquet', columns=['wid'])['wid'].to_numpy())
j = np.searchsorted(hist_wids, te['wid'].to_numpy())
te_cold = (j >= len(hist_wids)) | (hist_wids[np.minimum(j, len(hist_wids) - 1)] != te['wid'].to_numpy())
pos = te['label'].to_numpy() > 0
print(f'test positives that are cold items: {te_cold[pos].mean():.1%} '
      f'(vs {te_cold.mean():.1%} of all test candidates)')

cold_pos = pd.Series(np.where(pos, te_cold, np.nan), index=te.index)
seg_frac = cold_pos.groupby(te['user']).mean()                      # share of a slate's positives that are cold
seg_of_user = pd.cut(seg_frac, [-0.01, 0.0, 0.999, 1.0], labels=['all warm', 'mixed', 'all cold'])
seg = te['user'].map(seg_of_user)

rows = {}
for s in ['all warm', 'mixed', 'all cold']:
    m = (seg == s).to_numpy()
    rows[f'base — positives {s}'] = evaluate(te[m], base.predict(te.loc[m, FEATURES_BASE]))
    rows[f'+text — positives {s}'] = evaluate(te[m], full.predict(te.loc[m, FEATURES_ALL]))
pd.DataFrame(rows).T.round(4)

test positives that are cold items: 26.7% (vs 56.7% of all test candidates)


,NDCG@10,HR@10,Purchase-HR@10,slates
base — positives all warm,0.5038,0.8073,0.7481,18466.0
+text — positives all warm,0.5089,0.8119,0.7525,18466.0
base — positives mixed,0.2604,0.7419,0.4366,3379.0
+text — positives mixed,0.2664,0.7514,0.4479,3379.0
base — positives all cold,0.0820,0.1888,0.1240,3977.0
+text — positives all cold,0.1005,0.2271,0.1446,3977.0


## 7. Takeaways

- **The text features lift the benchmark from NDCG@10 0.4075 to 0.4144** (HR@10 0.705 → 0.714, Purchase-HR@10 0.663 → 0.670), with the base model reproducing Step 2's result exactly. `q_name_tfidf_sim` is the strongest of the four (3.8% of total split gain).
- **Alone they are weak (0.0955 vs 0.084 random)** — JD's retrieval already returns lexically relevant candidates, so *within a slate* text similarity barely separates items. The signal only emerges in combination with the behavioural features.
- **The lift concentrates exactly where it was predicted: cold candidates.** On the 3,977 test slates whose positives are all cold items (no history anywhere), NDCG@10 rises from 0.082 to 0.101 (+23% relative) and HR@10 from 0.189 to 0.227 (+20%). Warm slates barely move (0.504 → 0.509). This is the first feature group that reaches the cold 80% of the catalogue at all.
- **Cold slates remain the weak spot** (0.10 vs 0.50 warm) and are ~15% of test slates. Lexical TF-IDF only matches exact term IDs; the clear next lever is *semantic* matching — term embeddings trained on this corpus (e.g. word2vec over product names + queries, or a two-tower / sequence model that embeds the user history), which can match related-but-different terms.
- 26.7% of test *positives* are cold items (56.7% of all test candidates), so improving cold ranking has a real ceiling to claim.